STEP 0: Bring Groq's llm

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

STEP 1: Extract Text from PDF

In [2]:
# extracts text from pdf and make into langchain Documents (one page -> one Document)
# pypdf itself creates the metadata for each Document

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader("../docs/mospi_expert_review.pdf")

docs = loader.load()

/tmp/ipykernel_129457/3274459894.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/rithish/Desktop/projects/RAGForge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STEP 2: Convert the documents into chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

STEP 3: Create embedding model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings 

embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3481.96it/s]


STEP 4: Create and Store embeddings in vector db

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents( # langchain creates embeddings for those chunks and stores it in chromadb
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../vector_db" # embeddings are saved locally on this directory
)

STEP 5: Retrive top 3 chunks

In [6]:
vector_store.similarity_search("what is meant by CPI ?")

[Document(id='26fa5d60-056e-4932-b536-1ac813654bfd', metadata={'page_label': '7', 'creationdate': '2026-01-29T12:34:54+05:30', 'page': 6, 'total_pages': 258, 'moddate': '2026-01-29T12:36:04+05:30', 'producer': 'iLovePDF', 'creator': 'PyPDF', 'source': '../docs/mospi_expert_review.pdf'}, page_content='P a g e  | 1 \n \n \nChapter 1 \nIntroduction and Process adopted for Base Revision \n \nThe Consumer Price Index (CPI) is a measure of movement of prices in the items \nconsumed by the Households. The retail inflation based on CPI which provides year on \nyear changes in CPI is a key macro -economic indicator for measuring the health of any \neconomy. Across the world, CPI is considered a very well evolved indicator in terms of \nits concepts, definitions, methodology and usage.'),
 Document(id='896c5385-b212-40a6-ab08-9d1149824781', metadata={'total_pages': 258, 'page': 68, 'creator': 'PyPDF', 'page_label': '69', 'producer': 'iLovePDF', 'moddate': '2026-01-29T12:36:04+05:30', 'source': '